# Train a free local model to replace Jev

This notebook fine-tunes small decision models on the dataset made by `python -m scanner.cli export-dataset`
and checks them against the held-out test set:

1. **Zero-shot Laya**: how far an untrained decision model gets.
2. **Fine-tuned Laya** (`convaiinnovations/laya`, ~0.4B parameters).
3. **Fine-tuned Qwen3.5-0.8B** with an Unsloth decision head.

It then compares them, suggests accept / reject thresholds for `config/stages.yaml`, and uploads the
best model as a GGUF file for llama.cpp's decision server.

**Before you start**

- Runtime > Change runtime type > **T4 GPU** (free).
- Settings come from your `.env`: drag it into the Files pane on the left, or add the same names as
  Colab secrets (key icon). It needs `HF_DATASET_REPO`, `HF_MODEL_REPO` and a Hugging Face token
  (`HF_TOKEN`) that can read the dataset and write the model repo.
- Expect roughly 30 to 60 minutes in total on a T4.

Unsloth's decision-model API is new. If a cell fails, copy the error back to the conversation that
made this notebook.

In [4]:
%%capture
!pip install --upgrade unsloth python-dotenv

## Settings

In [5]:
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path(".env"), override=True)  # the .env you uploaded; re-run this cell after editing it


def setting(name, default=None):
    # .env or environment first, then Colab secrets.
    if os.getenv(name):
        return os.environ[name].strip()
    try:
        from google.colab import userdata
        return userdata.get(name).strip()
    except Exception:  # no such secret, or its "Notebook access" is off
        return default


HF_DATASET_REPO = setting("HF_DATASET_REPO")
HF_MODEL_REPO = setting("HF_MODEL_REPO")
HF_TOKEN = setting("HF_TOKEN")
assert HF_DATASET_REPO, "Set HF_DATASET_REPO in .env or as a Colab secret"
assert HF_MODEL_REPO, "Set HF_MODEL_REPO in .env or as a Colab secret"

# Checked now so a missing token fails here, not after training when the model is uploaded.
from huggingface_hub import whoami

assert HF_TOKEN, (
    "Set HF_TOKEN (a Hugging Face token with write access, https://huggingface.co/settings/tokens) "
    "in .env or as a Colab secret, with 'Notebook access' switched on"
)
os.environ["HF_TOKEN"] = HF_TOKEN
account = whoami(token=HF_TOKEN)  # a 401 here means the token itself is wrong or revoked
access = account.get("auth", {}).get("accessToken", {})
print(f"logged in to Hugging Face as {account['name']}, token {access.get('displayName')!r} ({access.get('role')})")
assert access.get("role") != "read", "This token is read-only: create one with write access"

MODELS = {
    "laya": "convaiinnovations/laya",
    "qwen": "unsloth/Qwen3.5-0.8B",
}
NEGATIVES_PER_POSITIVE = 10   # rejected papers trained on per relevant one
HARD_NEGATIVE_SHARE = 0.5     # of those, the share Jev found most borderline
EPOCHS = 2
SEED = 3407
PUBLIC_MODEL = True           # False uploads the trained model to a private repository
CHECK_NEGATIVES = 2000        # extra rejected papers to measure false accepts on
print(f"dataset {HF_DATASET_REPO} -> model {HF_MODEL_REPO}")

logged in to Hugging Face as matgolabek, token 'another for jev' (write)
dataset matgolabek/arxiv-uav-fpga-fusion -> model matgolabek/arxiv-uav-fpga-fusion-model


## Load the dataset

In [6]:
import json

from datasets import load_dataset

dataset = load_dataset(HF_DATASET_REPO)
train_rows = [dict(row) for row in dataset["train"]]
test_rows = [dict(row) for row in dataset["test"]] if "test" in dataset else []
assert test_rows, "test.jsonl is empty: export with --test-size 300 and push again"

# The relevance question is the first one in every row (export-dataset writes it first).
RELEVANCE = next(iter(json.loads(train_rows[0]["questions"])))


def gold_relevant(row):
    return json.loads(row["gold"])[RELEVANCE] is True


def jev_probability(row):
    return json.loads(row["jev"]).get(RELEVANCE, {}).get("probability")


for name, rows in (("train", train_rows), ("test", test_rows)):
    relevant = sum(gold_relevant(row) for row in rows)
    print(f"{name}: {len(rows):,} papers, {relevant} relevant")

README.md:   0%|          | 0.00/863 [00:00<?, ?B/s]

train.jsonl: reconstructing file:   0%|          |  0.00B / 80.5MB            

train.jsonl: downloading bytes:           |  0.00B            

test.jsonl:   0%|          | 0.00/1.59M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

train: 26,889 papers, 690 relevant
test: 300 papers, 150 relevant


## A balanced training set

Fewer than 1% of papers are relevant, so a model trained on everything learns to say no. Train on
every relevant paper plus `NEGATIVES_PER_POSITIVE` rejected ones each: half of those are the rejected
papers Jev found most borderline (they teach where the line is), the rest are random.

In [7]:
import random


def balanced(rows, negatives_per_positive, hard_share, seed):
    positives = [row for row in rows if gold_relevant(row)]
    negatives = [row for row in rows if not gold_relevant(row)]
    wanted = min(len(negatives), negatives_per_positive * len(positives))
    by_doubt = sorted(negatives, key=lambda row: jev_probability(row) or 0.0, reverse=True)
    hard = by_doubt[: int(wanted * hard_share)]
    rest = by_doubt[len(hard):]
    rng = random.Random(seed)
    chosen = positives + hard + rng.sample(rest, wanted - len(hard))
    rng.shuffle(chosen)
    return chosen


train_subset = balanced(train_rows, NEGATIVES_PER_POSITIVE, HARD_NEGATIVE_SHARE, SEED)
print(f"training on {len(train_subset):,} papers, {sum(map(gold_relevant, train_subset))} relevant")

# The test set is half relevant, but real scans are over 99% irrelevant, so a false-accept rate
# too small to show on 150 test papers still adds up over a year. Rejected papers the models never
# trained on measure it.
used = {row["id"] for row in train_subset}
unused_negatives = [row for row in train_rows if not gold_relevant(row) and row["id"] not in used]
check_negatives = random.Random(SEED).sample(unused_negatives, min(CHECK_NEGATIVES, len(unused_negatives)))
NEGATIVE_SHARE = sum(not gold_relevant(row) for row in train_rows) / len(train_rows)
print(f"{len(check_negatives):,} unseen rejected papers for measuring false accepts")

training on 7,590 papers, 690 relevant
2,000 unseen rejected papers for measuring false accepts


## How models are scored

Every test paper gets a relevance probability, split like the scanner does: at least `accept` is
accepted, at most `reject` is rejected, in between is "maybe" for you to review.

- **found**: share of relevant papers that are not rejected (accepted or maybe). Missed papers are
  the costly error, so this matters most.
- **precision**: share of accepted papers that are relevant.
- **maybe**: share of all papers left for you to review.
- **domain**: share of relevant papers whose sub-topic matches the gold answer.
- **false accepts**: share of unseen rejected papers the model accepts, and what that means per
  1,000 scanned papers at your real mix of relevant and irrelevant papers.

Test answers come from Jev unless you labelled the paper yourself, so on Jev-answered papers these
numbers measure agreement with Jev.

In [8]:
def yes_probability(answer):
    # Unsloth answers a yes/no question with the Decision API's "noul" probability.
    if "noul" in answer:
        return float(answer["noul"])
    probabilities = answer.get("probabilities", {})
    for key in (True, "true", "True", "yes"):
        if key in probabilities:
            return float(probabilities[key])
    raise ValueError(f"no yes probability in {answer}")


def score(probabilities, rows, accept=0.8, reject=0.2, domains=None):
    relevant = [gold_relevant(row) for row in rows]
    accepted = [p >= accept for p in probabilities]
    rejected = [p <= reject for p in probabilities]
    n_relevant = sum(relevant)
    result = {
        "found": sum(r and not x for r, x in zip(relevant, rejected)) / max(1, n_relevant),
        "precision": sum(r and a for r, a in zip(relevant, accepted)) / max(1, sum(accepted)),
        "maybe": sum(not a and not x for a, x in zip(accepted, rejected)) / len(rows),
    }
    if domains is not None:
        pairs = [(d, json.loads(row["gold"])) for d, row in zip(domains, rows) if gold_relevant(row)]
        pairs = [(d, gold) for d, gold in pairs if len(gold) > 1]
        hits = [d == next(v["label"] for k, v in gold.items() if k != RELEVANCE) for d, gold in pairs]
        result["domain"] = sum(hits) / len(hits) if hits else None
    return result


def suggest_thresholds(probabilities, rows, target_found=0.95, target_precision=0.9):
    # Highest reject threshold (at most 0.5) that still finds target_found of the relevant
    # papers, and the lowest accept threshold at least 0.1 above it whose accepted papers are
    # at least target_precision relevant. The limits keep a margin for papers unlike the test set.
    steps = [i / 100 for i in range(1, 100)]
    reject = max([t for t in steps if t <= 0.5 and score(probabilities, rows, 1.01, t)["found"] >= target_found], default=0.01)
    accept = min([t for t in steps if t >= reject + 0.1 and score(probabilities, rows, t, 0.0)["precision"] >= target_precision], default=0.99)
    return accept, reject


def false_accepts(negative_probabilities, accept=0.8):
    rate = sum(p >= accept for p in negative_probabilities) / max(1, len(negative_probabilities))
    return {"false accepts": rate, "per 1000 scanned": rate * NEGATIVE_SHARE * 1000}


def show(name, result):
    parts = [
        f"{key} {value:.1f}" if key == "per 1000 scanned" else f"{key} {value:.1%}"
        for key, value in result.items() if value is not None
    ]
    print(f"{name:<30}" + "   ".join(parts))


results = {}
human_test = [row for row in test_rows if row["label_source"] == "human"]
if human_test:
    show("Jev (your labels)", score([jev_probability(row) or 0.0 for row in human_test], human_test))
print(f"{len(test_rows)} test papers, {len(human_test)} with your own labels")

Jev (your labels)             found 100.0%   precision 0.0%   maybe 100.0%
300 test papers, 7 with your own labels


In [9]:
import torch
from unsloth import DecisionTrainer, FastDecisionModel
from transformers import TrainingArguments


def predict_all(model, tokenizer, rows):
    FastDecisionModel.for_inference(model)
    probabilities, domains = [], []
    for row in rows:
        answers = FastDecisionModel.predict(model, tokenizer, row["state"], json.loads(row["questions"]))
        probabilities.append(yes_probability(answers[RELEVANCE]))
        others = [answer for name, answer in answers.items() if name != RELEVANCE]
        domains.append(others[0].get("answer") if others else None)
    return probabilities, domains


def evaluate(name, model, tokenizer):
    probabilities, domains = predict_all(model, tokenizer, test_rows)
    negatives, _ = predict_all(model, tokenizer, check_negatives)
    results[name] = (probabilities, domains, negatives)
    show(name, score(probabilities, test_rows, domains=domains) | false_accepts(negatives))


def free_gpu():
    import gc
    gc.collect()
    torch.cuda.empty_cache()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## 1. Zero-shot Laya

In [10]:
model, tokenizer = FastDecisionModel.from_pretrained(MODELS["laya"], max_seq_length=512)
evaluate("laya zero-shot", model, tokenizer)
del model
free_gpu()

laya zero-shot                found 100.0%   precision 100.0%   maybe 97.7%   domain 16.0%   false accepts 0.1%   per 1000 scanned 1.0


## 2 and 3. Fine-tune Laya and Qwen

The same recipe for both, after Unsloth's `train_decision_from_lm.py`: LoRA, a few epochs on the
balanced set, then temperature calibration on a 10% holdout of it. Each trained model is saved to
`/content/models/<name>`.

In [11]:
def fine_tune(name, max_seq_length):
    model, tokenizer = FastDecisionModel.from_pretrained(
        MODELS[name], max_seq_length=max_seq_length, random_state=SEED,
    )
    items, report = FastDecisionModel.build_dataset(train_subset, tokenizer, model)
    print(f"{len(items):,} training decisions, {report['skipped']} skipped {report['reason'] or ''}")
    train, holdout = FastDecisionModel.split_holdout(items, SEED)

    model = FastDecisionModel.get_peft_model(model, r=32, lora_alpha=32, random_state=SEED)
    batch, accumulate = 8, 2
    steps = EPOCHS * -(-len(train) // (batch * accumulate))
    trainer = DecisionTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train,
        head_learning_rate=3e-4,
        args=TrainingArguments(
            output_dir=f"outputs/{name}",
            per_device_train_batch_size=batch,
            gradient_accumulation_steps=accumulate,
            learning_rate=1e-4,
            lr_scheduler_type="cosine",
            warmup_steps=max(1, steps // 20),
            weight_decay=0.01,
            num_train_epochs=EPOCHS,
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=10,
            report_to="none",
            save_strategy="no",
            seed=SEED,
        ),
    )
    trainer.train()
    if holdout:
        calibration = FastDecisionModel.calibrate(model, tokenizer, holdout)
        print({key: calibration[key] for key in ("accuracy", "ece") if key in calibration})

    evaluate(f"{name} fine-tuned", model, tokenizer)
    model.save_pretrained_merged(f"/content/models/{name}")
    del model, trainer
    free_gpu()

In [12]:
fine_tune("laya", max_seq_length=512)

Unsloth: 2,345 of 10,350 training inputs are longer than max_seq_length = 512, so the end of their state is cut. Raise max_seq_length to train on all of it.
10,350 training decisions, 0 skipped 
Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.326542
20,0.959869
30,0.668189
40,0.592966
50,0.477226
60,0.520349
70,0.487328
80,0.413839
90,0.487974
100,0.593039


{'accuracy': 0.7625, 'ece': 0.0}
laya fine-tuned               found 100.0%   precision 0.0%   maybe 100.0%   domain 62.7%   false accepts 0.0%   per 1000 scanned 0.0


ValueError: Unsloth: encoder.layers.0.attn.Wqkv.weight has NaN or values too large for float16, so the model cannot be saved.

In [ ]:
fine_tune("qwen", max_seq_length=1024)

## Compare

Scores at the scanner's default thresholds (accept 0.8, reject 0.2), then at thresholds tuned on the
test set for each model: find at least 95% of the relevant papers, and accept only when at least 90%
of accepted papers are relevant, with reject at most 0.5 and accept at least 0.1 above it. Tuned
thresholds fit the test set, so treat them as a starting point.

In [ ]:
best, best_key = None, None
for name, (probabilities, domains, negatives) in results.items():
    show(name, score(probabilities, test_rows, domains=domains) | false_accepts(negatives))
for name, (probabilities, domains, negatives) in results.items():
    accept, reject = suggest_thresholds(probabilities, test_rows)
    result = score(probabilities, test_rows, accept, reject, domains) | false_accepts(negatives, accept)
    show(f"{name} @ {accept:.2f}/{reject:.2f}", result)
    key = (result["found"], -result["false accepts"], result["precision"], -result["maybe"])
    if name.endswith("fine-tuned") and (best_key is None or key > best_key):
        best, best_key, best_thresholds = name.split()[0], key, (accept, reject)
print(f"\nbest: {best}, thresholds accept {best_thresholds[0]:.2f} reject {best_thresholds[1]:.2f}")

## Upload the best model as GGUF

Change `best` above if you prefer the other model. This converts it for llama.cpp's decision server
into `/content/gguf`, then uploads it to `HF_MODEL_REPO` (public unless `PUBLIC_MODEL = False`). If
the upload fails, fix the cause and run only the upload cell again; the conversion is kept. Put the
suggested thresholds into the relevance stage of `config/stages.yaml` when you switch the scanner to
the local model.

In [ ]:
model, tokenizer = FastDecisionModel.from_pretrained(f"/content/models/{best}")
model.save_pretrained_gguf("/content/gguf", tokenizer, quantization_method="q8_0")
print(sorted(str(path) for path in Path("/content/gguf").rglob("*.gguf")))
print(f"accept_threshold: {best_thresholds[0]:.2f}\nreject_threshold: {best_thresholds[1]:.2f}")

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
api.create_repo(HF_MODEL_REPO, private=not PUBLIC_MODEL, exist_ok=True)
api.upload_folder(
    repo_id=HF_MODEL_REPO,
    # Unsloth writes the files to <folder>/gguf.
    folder_path="/content/gguf/gguf" if Path("/content/gguf/gguf").is_dir() else "/content/gguf",
    commit_message=f"{best} decision model, accept {best_thresholds[0]:.2f} reject {best_thresholds[1]:.2f}",
)
print(f"uploaded {best} to https://huggingface.co/{HF_MODEL_REPO}")